In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.io import savemat

def solve_bad_boussinesq_paper_params():
    # ================= 1. 参数与计算区域 (与 PINN 训练区域一致) =================
    A_user = 0.02      # 初始高斯幅值
    b_user = 0.02      # 高斯衰减系数 (平方项内)

    L = 80.0           # 空间域 [-80, 80]  (与训练输出网格一致)
    t_max = 50.0       # 时间域 [0, 50]    (与训练输出网格一致)
    dt = 0.01          # 时间步长

    # 谱方法截断波数 N = floor(L / pi)
    N = int(np.floor(L / np.pi))  # N = 25
    M = 2 * N                     # ODE 维数 M = 50

    dx = 2.0 * L / M
    x = np.linspace(-L, L, M, endpoint=False)
    k = np.fft.fftfreq(M, d=dx) * 2.0 * np.pi

    # ================= 2. 高频阻尼平滑函数 d(j) =================
    d0 = 10.0
    Nd = int(np.floor(N / 8.0))

    def smooth_step(y):
        res = np.zeros_like(y)
        mask = (y > 0.0) & (y < 1.0)
        res[mask] = (y[mask]**4) * ((y[mask] - 2.0)**4)
        res[y >= 1.0] = 1.0
        return res

    j_arr = np.fft.fftfreq(M) * M
    d_j = np.zeros(M)
    for idx, j_val in enumerate(j_arr):
        if -N <= j_val <= -N + Nd:
            y = (j_val + N) / Nd
            d_j[idx] = d0 * (1.0 - smooth_step(y))
        elif N - 1 - Nd <= j_val <= N - 1:
            y = (j_val - N + Nd + 1) / Nd
            d_j[idx] = d0 * smooth_step(y)
        else:
            d_j[idx] = 0.0

    # ================= 3. 初值条件 =================
    u0 = 3.0 * A_user * np.exp(-b_user * (x**2))
    v0 = np.zeros_like(u0)

    u_hat = np.fft.fft(u0)
    v_hat = np.fft.fft(v0)

    # ================= 4. 输出网格 (与 pinnall_data.mat 完全一致: x 321 点, t 501 点) =================
    x_out = np.linspace(-L, L, 321)        # dx_out = 0.5
    t_out = np.linspace(0.0, t_max, 501)   # dt_out = 0.1

    def evaluate_on_grid(spec_hat):
        """把 M 点频谱零填充插值到 x_out 网格 (带限周期插值), 物理量 = 谱量/3"""
        P = len(x_out) - 1   # [-L, L) 上的点数, z=+L 处由周期性补齐
        spec_shift = np.fft.fftshift(spec_hat)
        pad_total = P - M
        pad_l = pad_total // 2
        pad_r = pad_total - pad_l
        spec_pad = np.pad(spec_shift, (pad_l, pad_r), mode='constant')
        vals = np.real(np.fft.ifft(np.fft.ifftshift(spec_pad))) * (P / M)
        return np.append(vals, vals[0]) / 3.0

    # 2D 切片图用的稠密网格
    dense_points = 2000
    x_dense = np.linspace(-L, L, dense_points, endpoint=False)

    def evaluate_continuous(spec_hat):
        """由频谱零填充还原物理量到稠密网格 (除 3)"""
        spec_shift = np.fft.fftshift(spec_hat)
        pad_total = dense_points - M
        pad_l = pad_total // 2
        pad_r = pad_total - pad_l
        spec_pad = np.pad(spec_shift, (pad_l, pad_r), mode='constant')
        spec_dense = np.real(np.fft.ifft(np.fft.ifftshift(spec_pad))) * (dense_points / M)
        return spec_dense / 3.0

    # ================= 5. ODE 傅里叶系数系统 (式 2.9) =================
    def rhs(u_h, v_h):
        u_phys = np.real(np.fft.ifft(u_h))
        u2_h = np.fft.fft(u_phys**2)

        du_h = 1j * k * v_h - d_j * u_h
        dv_h = 1j * k * u_h + 1j * k * u2_h - 1j * (k**3) * u_h
        return du_h, dv_h

    # ================= 6. RK4 时间推进, 按输出网格记录 =================
    n_steps = int(t_max / dt)          # 5000 步
    record_every = 10                  # 每 10 步记录一次 -> 输出 dt = 0.1

    t_snaps = [0.0, 20.0, 40.0, 50.0]  # 2D 切片图时刻
    saved_2d_h = {}
    saved_2d_v = {}

    H_out = np.zeros((len(t_out), len(x_out)))
    G_out = np.zeros((len(t_out), len(x_out)))

    # 记录 t=0
    H_out[0] = evaluate_on_grid(u_hat)
    G_out[0] = evaluate_on_grid(v_hat)
    saved_2d_h[0.0] = evaluate_continuous(u_hat)
    saved_2d_v[0.0] = evaluate_continuous(v_hat)

    print(f"开始求解并记录: L={L}, t_max={t_max}, M={M} ...")

    for step in range(1, n_steps + 1):
        k1_u, k1_v = rhs(u_hat, v_hat)
        k2_u, k2_v = rhs(u_hat + 0.5 * dt * k1_u, v_hat + 0.5 * dt * k1_v)
        k3_u, k3_v = rhs(u_hat + 0.5 * dt * k2_u, v_hat + 0.5 * dt * k2_v)
        k4_u, k4_v = rhs(u_hat + dt * k3_u, v_hat + dt * k3_v)

        u_hat += (dt / 6.0) * (k1_u + 2.0 * k2_u + 2.0 * k3_u + k4_u)
        v_hat += (dt / 6.0) * (k1_v + 2.0 * k2_v + 2.0 * k3_v + k4_v)

        # 按输出网格记录
        if step % record_every == 0:
            t_now = step * dt
            idx = step // record_every
            H_out[idx] = evaluate_on_grid(u_hat)
            G_out[idx] = evaluate_on_grid(v_hat)

            for t_s in t_snaps:
                if abs(t_now - t_s) < 0.5 * record_every * dt and t_s not in saved_2d_h:
                    saved_2d_h[t_s] = evaluate_continuous(u_hat)
                    saved_2d_v[t_s] = evaluate_continuous(v_hat)

    print("求解完成, 检查波包是否越界 ...")
    edge = np.abs(H_out[:, 0]).max()   # z = ±L 周期回绕点处的幅值
    print(f"|h(z=±{L:.0f}, t)| 最大值 = {edge:.3e}  (接近 0 说明波包未越界, 周期边界无污染)")

    # ================= 7. 保存参考 mat (与 pinnall_data.mat 完全相同的格式) =================
    X_mesh, T_mesh = np.meshgrid(x_out, t_out)
    savemat('bq_ref_data.mat', {
        'x': X_mesh.flatten()[:, None],
        't': T_mesh.flatten()[:, None],
        'h_ref': H_out.flatten()[:, None],
        'g_ref': G_out.flatten()[:, None],
    })
    print("已保存 bq_ref_data.mat: x/t/h_ref/g_ref 各 (160821,1)")

    # ================= 8. 绘图 =================

    # ---------------- 图 1: 不同时刻二维切片对比 ----------------
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5.5), dpi=120)

    for t_val, h_curve in saved_2d_h.items():
        ax1.plot(x_dense, h_curve, label=f'$t = {t_val:.1f}$', linewidth=1.8)
    ax1.set_xlim(-L, L)
    ax1.set_ylim(-0.005, 0.025)
    ax1.set_xlabel('$z$', fontsize=12)
    ax1.set_ylabel('$h(z, t)$', fontsize=12)
    ax1.set_title('Wave Elevation $h(z, t)$ at Specified Times', fontsize=13)
    ax1.grid(True, linestyle='--', alpha=0.6)
    ax1.legend(fontsize=10, loc='upper right')

    for t_val, v_curve in saved_2d_v.items():
        ax2.plot(x_dense, v_curve, label=f'$t = {t_val:.1f}$', linewidth=1.8)
    ax2.set_xlim(-L, L)
    ax2.set_xlabel('$z$', fontsize=12)
    ax2.set_ylabel(r'$v(z, t) / 3$', fontsize=12)
    ax2.set_title(r'Auxiliary Potential $v(z, t) / 3$ at Specified Times', fontsize=13)
    ax2.grid(True, linestyle='--', alpha=0.6)
    ax2.legend(fontsize=10, loc='upper right')

    plt.tight_layout()

    # ---------------- 图 2: h(z, t) 三维曲面 ----------------
    Z_mesh, T_mesh = np.meshgrid(x_out, t_out)

    fig2 = plt.figure(figsize=(10, 7), dpi=120)
    ax3d_1 = fig2.add_subplot(111, projection='3d')
    surf1 = ax3d_1.plot_surface(Z_mesh, T_mesh, H_out, cmap='viridis',
                                rstride=10, cstride=4, edgecolor='none', alpha=0.95)
    ax3d_1.set_xlabel('$z$', fontsize=11, labelpad=8)
    ax3d_1.set_ylabel('$t$', fontsize=11, labelpad=8)
    ax3d_1.set_zlabel('$h(z, t)$', fontsize=11)
    ax3d_1.set_title('3D Spatiotemporal Evolution of $h(z, t)$', fontsize=13, pad=15)
    ax3d_1.set_xlim(-L, L)
    ax3d_1.set_ylim(0, t_max)
    ax3d_1.view_init(elev=28, azim=-60)
    fig2.colorbar(surf1, ax=ax3d_1, shrink=0.55, aspect=12, pad=0.1)
    plt.tight_layout()

    # ---------------- 图 3: v(z, t)/3 三维曲面 ----------------
    fig3 = plt.figure(figsize=(10, 7), dpi=120)
    ax3d_2 = fig3.add_subplot(111, projection='3d')
    surf2 = ax3d_2.plot_surface(Z_mesh, T_mesh, G_out, cmap='coolwarm',
                                rstride=10, cstride=4, edgecolor='none', alpha=0.95)
    ax3d_2.set_xlabel('$z$', fontsize=11, labelpad=8)
    ax3d_2.set_ylabel('$t$', fontsize=11, labelpad=8)
    ax3d_2.set_zlabel(r'$v(z, t) / 3$', fontsize=11)
    ax3d_2.set_title(r'3D Spatiotemporal Evolution of $v(z, t) / 3$', fontsize=13, pad=15)
    ax3d_2.set_xlim(-L, L)
    ax3d_2.set_ylim(0, t_max)
    ax3d_2.view_init(elev=28, azim=-60)
    fig3.colorbar(surf2, ax=ax3d_2, shrink=0.55, aspect=12, pad=0.1)
    plt.tight_layout()

    plt.show()

if __name__ == '__main__':
    solve_bad_boussinesq_paper_params()
